This book is used to visually show which planning area and subzone of singapore does not have demographic data according to 2010, 2015 and 2020 census data

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import shape
from pathlib import Path
import os
from bs4 import BeautifulSoup
import re

In [2]:
CURRENT_DIRECTORY =  os.getcwd()
SRC_DIRECTORY = Path(CURRENT_DIRECTORY).parents[1]
print(SRC_DIRECTORY)

BASE_DATASET_PATH = Path(SRC_DIRECTORY).parents[0]
BASE_DATASET_PATH = BASE_DATASET_PATH / "datasets"
print(BASE_DATASET_PATH)

/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/src
/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets


In [3]:
def read_gpkg_file(filepath: str):
    """
    Returns a geospatial dataframe of the file specified in the filepath
    Returns None if the filepath given does not exist
    """
    if filepath.exists():
        geospatial_df = gpd.read_file(filepath)
    else:
        print(f"filepath given {filepath} does not exists. Aborting")
        return None

    print(geospatial_df["Description"][1])
    return geospatial_df

In [4]:
def extract_subzone_name(description_html: str):
    """
    Parses the HTML-like string in the Description column to extract 
    the value associated with the 'SUBZONE_N' key.
    
    Args:
        description_html (str): The string content from the 'Description' column 
                                (containing the HTML table).
        
    Returns:
        str: The extracted subzone name (e.g., 'BUKIT MERAH') or None if not found.
    """
    # 1. Handle NaN/Null values
    if pd.isna(description_html) or not isinstance(description_html, str):
        return None
        
    try:
        # 2. Use BeautifulSoup to parse the table structure
        soup = BeautifulSoup(description_html, 'html.parser')
        
        # 3. Find all table rows (<tr>)
        rows = soup.find_all('tr')
        
        # 4. Iterate through the rows to find the one containing 'SUBZONE_N'
        for row in rows:
            # Look for the header cell (<th>) with the exact text 'SUBZONE_N'
            header_cell = row.find('th', string='SUBZONE_N')
            
            if header_cell:
                # The value is in the corresponding data cell (<td>) in the same row
                subzone_name_tag = row.find('td')
                
                if subzone_name_tag:
                    return subzone_name_tag.text.strip()
                    
        return None # Return None if the key 'SUBZONE_N' is not found
        
    except Exception as e:
        # Catch any parsing errors
        print(f"Error parsing description for a feature: {e}")
        return None

In [5]:
def extract_pln_area_name(description_html):
    """
    Parses the HTML-like string in the Description column to extract 
    the value associated with the 'PLN_AREA_N' key.
    
    Args:
        description_html (str): The string content from the 'Description' column.
        
    Returns:
        str: The extracted planning area name (e.g., 'BUKIT MERAH') or None if not found.
    """
    # Check for NaN/missing values
    if pd.isna(description_html):
        return None
        
    try:
        # Use BeautifulSoup to parse the table structure
        soup = BeautifulSoup(description_html, 'html.parser')
        
        # Find all table rows (<tr>)
        rows = soup.find_all('tr')
        
        # Iterate through the rows to find the one containing 'PLN_AREA_N'
        for row in rows:
            # Look for the cell containing the attribute key 'PLN_AREA_N'
            # Note: The structure is <th>PLN_AREA_N</th><td>BUKIT MERAH</td>
            # The 'th' tag usually holds the attribute name.
            if row.find('th', string='PLN_AREA_N'):
                # The value is in the sibling <td> cell
                pln_area_name_tag = row.find('td')
                if pln_area_name_tag:
                    return pln_area_name_tag.text.strip()
                    
        return None # Return None if the key 'PLN_AREA_N' is not found
        
    except Exception as e:
        # It's good practice to catch parsing errors
        print(f"Error parsing description for a feature: {e}")
        return None

In [6]:
def add_subzone_pln_area_name(subzone_filepath, output_file, add_subzone_name = False, add_pln_area_name = False):
    subzone_df = read_gpkg_file(subzone_filepath)
    if subzone_df is not None:
        display(subzone_df.head())
    
    if add_subzone_name:
        subzone_df["SUBZONE_N"] = subzone_df["Description"].apply(extract_subzone_name)
    if add_pln_area_name:
        subzone_df["PLN_AREA_N"] = subzone_df["Description"].apply(extract_pln_area_name)

    subzone_df.to_file(
        output_file,
        layer = "subzone",
        driver = "GPKG",
        mode = "w" # Use 'w' (write mode) to overwrite the existing layer
    )
    print("Done")



In [7]:
def save_to_excel_with_multiple_sheets(df, filepath, sheet_name_to_update= None):
    if filepath.exists():
        excel_mode = 'a'
        writer_kwargs = {'mode': excel_mode, 'if_sheet_exists': 'replace'}
    else:
        excel_mode = 'w'
        writer_kwargs = {'mode': excel_mode}

    with pd.ExcelWriter(
        filepath, 
        engine='openpyxl',
        **writer_kwargs # Unpack the dictionary of arguments
    ) as writer:
        # Write the DataFrame to the specified sheet
        df.to_excel(
            writer, 
            sheet_name=sheet_name_to_update, 
            index=False # Set to True if you want to save the DataFrame index
        )
    print(f"Save to filepath: {filepath}" )

In [8]:
subzone_filepath = Path(BASE_DATASET_PATH / "singapore_data/data_gov/masterplan_2019/MasterPlan2019SubzoneBoundary.gpkg")
save_to_filepath = Path(BASE_DATASET_PATH / "singapore_data/data_gov/masterplan_2019/subzone_boundary_2019.gpkg")
add_subzone_pln_area_name(subzone_filepath, save_to_filepath, True, True)

<center><table><tr><th colspan='2' align='center'><em>Attributes</em></th></tr><tr bgcolor="#E3E3F3"> <th>SUBZONE_NO</th> <td>2</td> </tr><tr bgcolor=""> <th>SUBZONE_N</th> <td>BUKIT MERAH</td> </tr><tr bgcolor="#E3E3F3"> <th>SUBZONE_C</th> <td>BMSZ02</td> </tr><tr bgcolor=""> <th>CA_IND</th> <td>N</td> </tr><tr bgcolor="#E3E3F3"> <th>PLN_AREA_N</th> <td>BUKIT MERAH</td> </tr><tr bgcolor=""> <th>PLN_AREA_C</th> <td>BM</td> </tr><tr bgcolor="#E3E3F3"> <th>REGION_N</th> <td>CENTRAL REGION</td> </tr><tr bgcolor=""> <th>REGION_C</th> <td>CR</td> </tr><tr bgcolor="#E3E3F3"> <th>INC_CRC</th> <td>085EF219A5A1AEAD</td> </tr><tr bgcolor=""> <th>FMEL_UPD_D</th> <td>20191223152313</td> </tr></table></center>


,Name,Description,geometry
0,kml_1,<center><table><tr><th colspan='2' align='cent...,"POLYGON Z ((103.81454 1.28239 0, 103.81774 1.2..."
1,kml_2,<center><table><tr><th colspan='2' align='cent...,"POLYGON Z ((103.82209 1.28049 0, 103.8221 1.28..."
2,kml_3,<center><table><tr><th colspan='2' align='cent...,"POLYGON Z ((103.84375 1.28508 0, 103.844 1.284..."
3,kml_4,<center><table><tr><th colspan='2' align='cent...,"POLYGON Z ((103.84962 1.28412 0, 103.84955 1.2..."
4,kml_5,<center><table><tr><th colspan='2' align='cent...,"POLYGON Z ((103.85253 1.28617 0, 103.85253 1.2..."


Done


#### checking for planning areas that are not included in the 2010, 2015 and 2020 dataset

In [9]:
def get_relevant_dataframes_pln_area(population_year: int, geospatial_year: int):
    pln_area_filepath = Path(BASE_DATASET_PATH / f"singapore_data/onemap_data/planning_areas_{geospatial_year}.gpkg")
    excel_filepath = Path(BASE_DATASET_PATH /  f"singapore_data/cleaned_data/{population_year}_age_group_planning_area_subzone.xlsx")

    pln_area_df = gpd.read_file(pln_area_filepath)
    # make the column names of the pln_area dataframe small caps
    pln_area_df.columns = [x.lower() for x in pln_area_df]
    # convert the names 
    # pln_area_df["subzone_n"] = pln_area_df["subzone_n"].str.strip().str.lower()
    pln_area_df["pln_area_n"] = pln_area_df["pln_area_n"].str.strip().str.lower()

    population_age_group_df = pd.read_excel(excel_filepath, sheet_name = "planning_area")
    population_age_group_df.columns = [x.lower() for x in population_age_group_df]

    return pln_area_df, population_age_group_df

In [10]:
def merge_pln_area_demographics_with_geospatial(population_year: int, geospatial_year: int):
    """
    check if the census data is missing out on any demographics data for each planning area in the geospatial data
    """
    pln_area_df, population_age_group_df = get_relevant_dataframes_pln_area(population_year, geospatial_year)

    # Rename the column 'pln_area_n' to 'planning_area' for a clean merge with the population data.
    pln_area_df = pln_area_df.rename(columns={'pln_area_n': 'planning_area'})

    # Merge the population data with the planning area geometries
    merged_gdf = pln_area_df.merge(
        population_age_group_df,
        on='planning_area', # The common key for joining
        how='inner'         # Use 'inner' to include only areas present in both dataframes
    )

    # Set the result as a GeoDataFrame (the merge might cast it back to a DataFrame)
    merged_gdf = gpd.GeoDataFrame(merged_gdf, geometry='geometry')

    # display(merged_gdf.head())
    # display(merged_gdf.info())

    output_path = Path(BASE_DATASET_PATH / f"singapore_data/visualisation_data/pln_area_with_population_data_{population_year}.gpkg")
    merged_gdf.to_file(output_path)

    print("Done!")

In [11]:
def run_for_all_years(year_pairs: dict):
    for population_year, geospatial_year in year_pairs.items():
        merge_pln_area_demographics_with_geospatial(population_year, geospatial_year)

In [12]:
year_pairs = {
    2020: 2019,
    2015: 2014,
    2010: 2008,
}

run_for_all_years(year_pairs)

Done!
Done!
Done!


In [13]:
def get_relevant_dataframes_subzone(population_year: int, geospatial_year: int):
    subzone_filepath = Path(BASE_DATASET_PATH / f"singapore_data/data_gov/masterplan_{geospatial_year}/subzone_boundary_{geospatial_year}.gpkg")
    excel_filepath = Path(BASE_DATASET_PATH /  f"singapore_data/cleaned_data/{population_year}_age_group_planning_area_subzone.xlsx")

    subzone_df = gpd.read_file(subzone_filepath)
    # make the column names of the pln_area dataframe small caps
    subzone_df.columns = [x.lower() for x in subzone_df]
    # convert the names 
    subzone_df["subzone_n"] = subzone_df["subzone_n"].str.strip().str.lower()
    subzone_df["pln_area_n"] = subzone_df["pln_area_n"].str.strip().str.lower()

    population_age_group_df = pd.read_excel(excel_filepath, sheet_name = "subzone")
    population_age_group_df.columns = [x.lower() for x in population_age_group_df]

    ## fill the empty cells with the planning area names so that it is easier to work with
    # If the empty cells were loaded as empty strings instead of NaN (missing value marker),
    # replace them with NaN first so fillna can work correctly.
    population_age_group_df['planning_area'] = population_age_group_df['planning_area'].replace('', pd.NA)
    # Use the 'ffill' (Forward Fill) method
    # This propagates the last valid 'planning_area' value downward.
    population_age_group_df['planning_area'] = population_age_group_df['planning_area'].ffill()

    # display(population_age_group_df[population_age_group_df["subzone"] == "victoria"])

    return subzone_df, population_age_group_df

In [14]:
def merge_subzone_demographics_with_geospatial(population_year: int, geospatial_year: int,
                                               overwrite_subzone_df = None,
                                               overwrite_population_df = None):
    """
    check if the census data is missing out on any demographics data for each planning area in the geospatial data
    """
    subzone_df, population_age_group_df = get_relevant_dataframes_subzone(population_year, geospatial_year)

    if overwrite_subzone_df is not None:
        subzone_df = overwrite_subzone_df
    if overwrite_population_df is not None:
        population_age_group_df = overwrite_population_df

    # Rename the column 'pln_area_n' to 'planning_area' for a clean merge with the population data.
    subzone_df = subzone_df.rename(columns={'pln_area_n': 'planning_area', "subzone_n": "subzone"})
    # subzone_df = subzone_df.rename(columns={'subzone_n': 'subzone'})

    # Merge the population data with the planning area geometries
    merged_gdf = subzone_df.merge(
        population_age_group_df,
        on=['planning_area', 'subzone'], # The common key for joining
        how='inner'         # Use 'inner' to include only areas present in both dataframes
    )

    # Set the result as a GeoDataFrame (the merge might cast it back to a DataFrame)
    merged_gdf = gpd.GeoDataFrame(merged_gdf, geometry='geometry')

    # display(merged_gdf.head())
    # display(merged_gdf.info())

    output_path = Path(BASE_DATASET_PATH / f"singapore_data/visualisation_data/subzone_with_population_data_{population_year}.gpkg")
    merged_gdf.to_file(output_path)

    # print("Done!")

    return merged_gdf

In [15]:
# merged_gdf = merge_subzone_demographics_with_geospatial(2010, 2008)

In [16]:
def find_missing_subzones(population_year: int, geospatial_year: int, merged_gdf):
    """
    outputs a csv file of subzone planning_area pair that is not captured in the merged_gdf df
    subzones with the name total will not be present in merged_gdf,
    hence only subzones with the name "total" should appear in the missing_subzones csv
    """
    subzone_df, population_age_group_df = get_relevant_dataframes_subzone(population_year, geospatial_year)

    subzone_df_keys = subzone_df.rename(columns={'pln_area_n': 'planning_area', 'subzone_n': 'subzone'})
    # subzone_df_keys = subzone_df_keys[['planning_area', 'subzone']].drop_duplicates()

    # 2. Perform a left merge: Keep all rows from population_age_group_df (left)
    #    and mark whether they found a match in subzone_df (right).
    missing_data_df = population_age_group_df.merge(
        subzone_df_keys,
        on=['planning_area', 'subzone'],
        how='left',
        indicator=True
    )

    # 3. Filter for rows that only exist on the left side (population data)
    unmerged_pairs = missing_data_df[missing_data_df['_merge'] == 'left_only']
    missing_subzone_population_pairs = unmerged_pairs[['planning_area', 'subzone']].drop_duplicates()
    output_path = Path(BASE_DATASET_PATH / f"singapore_data/visualisation_data/missing_subzones.csv")
    missing_subzone_population_pairs.to_csv(output_path)

    return missing_subzone_population_pairs

In [17]:
# missing_subzone_population_pairs = find_missing_subzones(2010, 2008, merged_gdf)

### Checking if 2015 demographic data has any missing subzones from 2014 masterplan geospatial data

#### checking the list of missing subzone planning area pairs, it was observed that victoria was wrongly classified under jurong east, when it should be under rochor -> https://en.wikipedia.org/wiki/Rochor
Seems like the data from data.gov placed victoria under the wrong planning area

Changing the planning area for victoria to rochor instead of jurong east

In [20]:
subzone_df, population_age_group_df = get_relevant_dataframes_subzone(2015, 2014)

population_age_group_df.loc[
    population_age_group_df["subzone"] == "victoria", 
    "planning_area"
] = "rochor"

population_age_group_df = population_age_group_df.rename(columns = {
    "total_total": "total"
})

# save the changes to the excel file
save_to_filepath = Path(BASE_DATASET_PATH /  f"singapore_data/cleaned_data/2015_age_group_planning_area_subzone.xlsx")
save_to_excel_with_multiple_sheets(population_age_group_df, save_to_filepath, "subzone")

merged_gdf_edited = merge_subzone_demographics_with_geospatial(2015, 2014,
                                                               overwrite_population_df = population_age_group_df)

display(merged_gdf_edited[merged_gdf_edited["subzone"] == "victoria"])

Save to filepath: /Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets/singapore_data/cleaned_data/2015_age_group_planning_area_subzone.xlsx


,objectid,subzone_no,subzone,subzone_c,ca_ind,planning_area,pln_area_c,region_n,region_c,inc_crc,...,females_45_49,females_50_54,females_55_59,females_60_64,females_65_69,females_70_74,females_75_79,females_80_84,females_85andover,total_above_60
130,131,10,victoria,RCSZ10,Y,rochor,RC,CENTRAL REGION,CR,DD1436D810825828,...,60,80,110,90,100,60,60,30,30,660


### Checking if 2020 demographic data has any missing subzones from 2019 masterplan geospatial data

In [19]:
subzone_df, population_age_group_df = get_relevant_dataframes_subzone(2020, 2019)

# during my previous data cleaning, i removed the spacing in "southern islands". Rectifying the mistake
population_age_group_df.loc[
    population_age_group_df["planning_area"] == "southernislands", 
    "planning_area"
] = "southern islands"

# output_path = Path(BASE_DATASET_PATH / f"singapore_data/visualisation_data/subzone_df.csv")
# subzone_df.to_csv(output_path)
# output_path = Path(BASE_DATASET_PATH / f"singapore_data/visualisation_data/population_age_group_df.csv")
# population_age_group_df.to_csv(output_path)

merged_gdf = merge_subzone_demographics_with_geospatial(2020, 2019,
                                                        overwrite_population_df = population_age_group_df)
# missing_subzone_population_pairs = find_missing_subzones(2020, 2019, merged_gdf)

### Checking if 2010 demographic data has any missing subzones from 2008 masterplan geospatial data

In [20]:
subzone_df, population_age_group_df = get_relevant_dataframes_subzone(2010, 2008)

## there see to be a regex issue with shangri-la that i couldnt fix,manually overwriting it
population_age_group_df.loc[
    population_age_group_df["subzone"] == "shangritotalla", 
    "subzone"
] = "shangri-la"

## the subzones for punggol is named differently in the masterplan, instead of eg: subzone 2, it is sz2
population_age_group_df.loc[
    (population_age_group_df["subzone"] == "subzone 2") &
    (population_age_group_df["planning_area"] == "punggol"), 
    "subzone"
] = "sz2"
population_age_group_df.loc[
    (population_age_group_df["subzone"] == "subzone 3") &
    (population_age_group_df["planning_area"] == "punggol"), 
    "subzone"
] = "sz3"
population_age_group_df.loc[
    (population_age_group_df["subzone"] == "subzone 4") &
    (population_age_group_df["planning_area"] == "punggol"), 
    "subzone"
] = "sz4"
population_age_group_df.loc[
    (population_age_group_df["subzone"] == "subzone 5") &
    (population_age_group_df["planning_area"] == "punggol"), 
    "subzone"
] = "sz5"

merged_gdf = merge_subzone_demographics_with_geospatial(2010, 2008,
                                                        overwrite_population_df = population_age_group_df)
missing_subzone_population_pairs = find_missing_subzones(2010, 2008, merged_gdf)